In [ ]:
import json
import numpy as np
import pymbar
import matplotlib.pyplot as plt
import os

## Trajectories and Scores classes

We define two classes to store our generated data in:
* Trajectories: stores the generated trajectories and annealing schedules for all of the trajectories. 
* Scores: Stores samples that will be considered unordered, such as those generated through unbiased sampling. 

In [ ]:
DATA_LOAD_DIR_MAIN = os.path.join("data", "paper")
DATA_LOAD_DIR_EXAMPLE = os.path.join("data", "example")

USE_EXAMPLE_DATA = False
#obs_name = "ari" 
obs_name = "log_probs"

if USE_EXAMPLE_DATA:
    DATA_LOAD_DIR = DATA_LOAD_DIR_EXAMPLE
else:
    DATA_LOAD_DIR = DATA_LOAD_DIR_MAIN


trajs_dir = os.path.join(DATA_LOAD_DIR, f"{obs_name}_trajectories.json")
unbiased_dir = os.path.join(DATA_LOAD_DIR, f"{obs_name}_unbiased_samples.json")


In [ ]:
class Trajectories():
    def __init__(self, trajectories: list[list[float]], 
                biases: list[list[float]], 
                steps_per_bias: float):
        """Initializes trajectory object.

        Args:
            trajectories (list[list[float]]): A list of trajectories.
            biases (list[list[float]]): A list of the biases used in each trajectory.
            steps_per_bias (list[list[int]]): A list of the number of steps taken at each bias in each trajectory.
        """
        self.trajectories = trajectories
        self.biases = biases
        self.steps_per_bias = steps_per_bias
        
    def get_all_biases(self) -> list[float]:
        """Returns an ordered list of all the biases in the trajectories object."""
        all_biases = []
        for biases_per_traj in self.biases: 
            for bias in biases_per_traj:
                if bias not in all_biases:
                    all_biases.append(bias)
                    
        return sorted(all_biases)
    
    def get_all_trajs_per_bias(self) -> tuple[float, list[np.array]]: 
        """Returns a the ordered list of biases, and a corresponding list of b x steps_per_bias numpy arrays, where b is the total number of annealing steps that use that bias.  
        """
        all_biases = []
        all_trajs_per_bias = []
        for biases_for_trajs, trajs_per_biases in zip(self.biases, self.trajectories): 
            cum_steps = 0
            for bias in biases_for_trajs:
                if bias not in all_biases:
                    all_biases.append(bias)
                    all_trajs_per_bias.append([])
                    for traj in trajs_per_biases:
                        all_trajs_per_bias[-1].append(traj[cum_steps : cum_steps + self.steps_per_bias])
                else:
                    for traj in trajs_per_biases:
                        index = all_biases.index(bias)
                        all_trajs_per_bias[index].append(traj[cum_steps : cum_steps + self.steps_per_bias])
                
                cum_steps += self.steps_per_bias
                
        sorted_indices = np.argsort(all_biases)
        all_biases = [all_biases[i] for i in sorted_indices]
        all_trajs_per_bias = [all_trajs_per_bias[i] for i in sorted_indices]
        return all_biases, np.array(all_trajs_per_bias)
    
    def get_scores(self):
        all_biases, all_trajs_per_bias_np = self.get_all_trajs_per_bias()
        scores_dict = {}
        for bias, trajs_per_bias_np in zip(all_biases, all_trajs_per_bias_np):
            scores_dict[bias] = trajs_per_bias_np.flatten().tolist()
            
        return Scores(scores_dict)
        
class Scores():
    def __init__(self, scores_dict: dict):
        """Initializes the Scores object.
        
        Args: 
            scores_dict (dict): A dictionary where keys are floats, denoting the bias, and the values are lists of scores for that bias value. 
        
        """
        self.scores_dict = scores_dict
    
    def __add__(self, other: "Scores") -> "Scores":
        new_scores_dict = self.scores_dict
        for key, value in other.scores_dict.items():
            if key in new_scores_dict.keys():
                new_scores_dict[key].extend(value)
            else:
                new_scores_dict[key] = value

        return Scores(new_scores_dict)
    
    def get_all_biases(self) -> list[float]:
        return sorted(self.scores_dict.keys())

In [ ]:
# with open(trajs_dir, "r") as f: 
#     trajectories_saved = json.load(f)
    
# new_trajectories = []
# new_biases = []

# for key, trajs in trajectories_saved.items():
#     biases, steps_per_bias = eval(key)
#     new_biases.append(biases)
#     new_trajectories.append(trajs)
    
# new_steps_per_bias = steps_per_bias[0]

# with open("data/paper/ari_trajectories_new.json", "w") as f: 
#     json.dump([new_trajectories, new_biases, new_steps_per_bias], f)

In [ ]:
# Load the saved trajectories
with open(trajs_dir, "r") as f: 
    trajectories_saved = json.load(f)
    trajectories = Trajectories(trajectories_saved[0], trajectories_saved[1], trajectories_saved[2])

with open(unbiased_dir, "r") as f: 
    unbiased_scores_list = json.load(f)
    unbiased_scores = Scores({0 : unbiased_scores_list})

## Removing Unconverged Data
We use two methods to remove unconverged data. The first is burnin, which eliminates the start of the chain, where it is still approaching the target distribution, and the second is by eliminating any biases which are determined to be unconverged, with respect to the Gelman Rubin statistic.

### Burnin
This allows us to discard the unconverged part of each annealing step, for each trajectory.

In [ ]:
def apply_burnin(trajectories: Trajectories, burnin: float=0.1) -> Trajectories:
    """Applies burnin to the trajectory, per annealing step."""
    burnin_steps = int(trajectories.steps_per_bias * burnin)
    new_trajectories = []
    for biases, trajs_per_biases in zip(trajectories.biases, trajectories.trajectories): 
        new_trajs_per_biases = []
        for traj in trajs_per_biases:
            new_traj = []
            for i in range(len(biases)):
                new_traj += traj[i * trajectories.steps_per_bias + burnin_steps : (i + 1) * trajectories.steps_per_bias]
            
            new_trajs_per_biases.append(new_traj)
        new_trajectories.append(new_trajs_per_biases) 
            
    
    return Trajectories(new_trajectories, trajectories.biases, trajectories.steps_per_bias - burnin_steps)

trajectories_burnin = apply_burnin(trajectories, 0.1)

### Gelman Rubin
We compute the Gelman-Rubin statistic as a measure of convergence. This gives us a unique value for each bias value, determining how well converged the chains from this bias value are.

The function returns a scores object with only the converged bias values in it, as well as a list of all GR values, ordered from lowest to highest bias. 

WARNING: If testing with small TPS runs, all data may be unconverged, meaning all data is removed. This will cause later code to error.

In [ ]:
def gelman_rubin(trajectories: Trajectories, cutoff: float=1.1) -> tuple[Scores, list]:
    biases, all_trajs_per_bias = trajectories.get_all_trajs_per_bias()
    
    grs = []
    scores_dict = {}
    for bias, trajs_per_bias in zip(biases, all_trajs_per_bias):
        j, L = trajs_per_bias.shape
        
        means = np.mean(trajs_per_bias, axis=1)
        mean_of_means = np.mean(means)
        B = L / (j - 1) * np.sum((means - mean_of_means) ** 2)
        W = 1 / j * np.sum(np.var(trajs_per_bias, axis=1, ddof=1))
        var_hat = (L - 1) / L * W + B / L
        R_hat = np.sqrt(var_hat / W)
        grs.append(R_hat)
        
        if R_hat < cutoff: 
            scores_dict[bias] = trajs_per_bias.flatten().tolist()
    
    scores = Scores(scores_dict)        
    return scores, grs

accepted_scores, grs = gelman_rubin(trajectories_burnin)

In [ ]:
plt.plot(trajectories.get_all_biases(), np.array(grs) - 1, label="GR")
plt.axhline(0.1, linestyle="--", linewidth=1, color="grey", label="cutoff")
plt.xlabel("$\lambda$")
plt.ylabel("GR")
plt.yscale("log")
plt.legend()

## Including Unbiased Samples

We also need to add in some samples from the unbiased distribution. We can do this by subsampling the biased distribution. 

We sample steps_per_bias * N samples / 2, where N is the number of trajectories we ran for each annealing schedule. This makes the number of tokens generated for each bias (including the unbiased, i.e. $\lambda = 0$) fixed, since unbiased completions require, on average, twice as many token generations as the biased ones. 

We will also initialize the list of samples for the unbiased histogram too. 

In [ ]:
def subsample_scores(scores, num_samples: int, random=False):
    subsample_dict = {}
    if random==True:
        for bias in scores.get_all_biases():
            subsample_dict[bias] = np.random.choice(scores.scores_dict[bias], num_samples).tolist()
    else:
        for bias in scores.get_all_biases():
            subsample_dict[bias] = scores.scores_dict[bias][:num_samples]
            
    return Scores(subsample_dict)

# We want to use trajectories here, not trajectories_burnin, since we want to match the number of generated samples. 

num_trajs_per_schedule = len(trajectories.trajectories[0])
num_biases_per_anneal = len(trajectories.biases[0])

unbiased_samples_for_biased = int(trajectories.steps_per_bias * num_trajs_per_schedule / 2)
accepted_scores += subsample_scores(unbiased_scores, num_samples = unbiased_samples_for_biased)

# Number of samples to use for the unbiased histogram. 
num_samples_for_unbiased = trajectories.steps_per_bias * num_trajs_per_schedule * num_biases_per_anneal // 2 + unbiased_samples_for_biased
unbiased_samples_for_histogram = unbiased_scores_list[:num_samples_for_unbiased]

## Reweighting the Samples

### MBAR
We use the Pymbar package to compute the Multistate Bennett Acceptance Ratio estimate of the partition function values at each bias. We can also use this package to calculate the overlap between the distributions too. 

In [ ]:
def mbar(scores, return_overlap = False) -> list[float]:
    biases = scores.get_all_biases()
    try:
        zero_index = biases.index(0)
    except:
        raise Exception("0 must be in the scores dictionary to perform mbar.")
    all_scores = np.array([])
    Ns = np.array([])
    for bias in biases:
        all_scores = np.append(all_scores, scores.scores_dict[bias])
        Ns = np.append(Ns, len(scores.scores_dict[bias]))
        
    u_kn = np.outer(biases, all_scores)
    mbar = pymbar.MBAR(u_kn, Ns)
    mbar_results = mbar.compute_free_energy_differences()
    
    res = (- mbar_results["Delta_f"][zero_index]).tolist()
    if return_overlap:
        overlap = mbar.compute_overlap() 
        return res, overlap
    else:
        return res
    
log_Zs = mbar(accepted_scores)

In [ ]:
plt.plot(accepted_scores.get_all_biases(), log_Zs)
plt.xlabel("$\lambda$")
plt.ylabel("$\log Z(\lambda)$")

### Calculating the Importance Sampling Weights

In [ ]:
def get_weights(scores: Scores, log_Zs):
    scores_list = []
    weights_list = []
    
    for i, bias in enumerate(scores.get_all_biases()):
        scores_list += scores.scores_dict[bias]
        weights_list += np.exp(log_Zs[i] + bias * np.array(scores.scores_dict[bias])).tolist()
        
    return scores_list, weights_list

scores_list, weights_list = get_weights(accepted_scores, log_Zs)

## Histograms

In [ ]:
"""Uncomment below to recreate paper for ARI:"""
#lower, upper, num_bins = (-8, 15, 80)

"""Uncomment below to recreate paper for logprobs:"""
# lower, upper, num_bins = (-600, 0, 100)

"""Minimal example:"""
lower, upper, num_bins = (0, 8, 10)

In [ ]:
bins = np.linspace(lower, upper, num_bins)
bin_centers = bins[:-1] + (bins[1:] - bins[:-1]) / 2

histogram_heights, _ = np.histogram(scores_list, weights=weights_list, bins=bins, density=True)

unbiased_histogram_heights, _ = np.histogram(unbiased_samples_for_histogram, bins=bins, density=True)

In [ ]:
plt.step(bin_centers, histogram_heights, where="mid", label="Biased", zorder=1)
plt.step(bin_centers, unbiased_histogram_heights, where="mid", label="Unbiased", zorder=0)
plt.xlabel(obs_name)
plt.ylabel("Density")
plt.yscale("log")
plt.legend()

## Error Estimates

### Bootstrapping
For the biased histogram, we compute a confidence interval using bootstrapping.

In [ ]:
def generate_trajectories_bootstrap(trajectories: Trajectories) -> Trajectories:
        """Generates a new Trajectory object, by bootstrapping over the trajectories.
        Returns:
            Trajectories: A new Trajectories object with bootstrapped data.
        """
        new_trajectories = []
        for biases, trajs in zip(trajectories.biases, trajectories.trajectories):
           indices = np.random.randint(0, len(trajs), len(trajs))
           new_trajectories.append([trajs[i] for i in indices])
            
        return Trajectories(new_trajectories, trajectories.biases, trajectories.steps_per_bias)
    
def bootstrap_hist_heights(trajectories: Trajectories, 
                           unbiased_scores: Scores, 
                           bins: np.array, 
                           num_bootstraps: int=100):
    trajs_burnin = apply_burnin(trajectories)
    
    num_unbiased = len(unbiased_scores.scores_dict[0]) // 2
    
    boot_histogram_heights = np.zeros((num_bootstraps, len(bins) - 1))
    for i in range(num_bootstraps):
        bootstrap_trajs = generate_trajectories_bootstrap(trajs_burnin)
        accepted_scores, _ = gelman_rubin(bootstrap_trajs)
        accepted_scores += subsample_scores(unbiased_scores, num_unbiased, random=True)    
        log_Zs = mbar(accepted_scores)
        scores_list, weights_list = get_weights(accepted_scores, log_Zs)

        histogram_heights, _ = np.histogram(scores_list, weights=weights_list, bins=bins, density=True)
        boot_histogram_heights[i, :] = histogram_heights
        
    return boot_histogram_heights
        
def bootstrap_conf_interval(trajectories: Trajectories, 
                           unbiased_scores: Scores, 
                           bins: np.array, 
                           num_bootstraps: int=100,
                           lower_idx: int=2,
                           upper_idx: int=97):
    
    if upper_idx > num_bootstraps - 1: 
        raise Exception("upper_idx must be less than or equal to num_bootstraps - 1.")
    if lower_idx >= upper_idx: 
        raise Exception("lower_idx must be strictly less than upper_idx.")
    
    boot_heights = bootstrap_hist_heights(trajectories, unbiased_scores, bins, num_bootstraps)
    
    sorted_boot_heights = np.sort(boot_heights, axis=0)
    lower_bounds = sorted_boot_heights[lower_idx]
    upper_bounds = sorted_boot_heights[upper_idx]
    return lower_bounds, upper_bounds    

WARNING: Note that this next cell will take a long time to run. With the "quick example" parameters, this took 17 minutes to run on my desktop, so expect this to be several hours for 100 bootstraps. It is recommended to parallelize this if using a higher number of bootstraps. 

In [ ]:
"""Uncomment below to recreate bootstrapping estimates from the paper."""
# num_bootstraps = 100
# lower_idx = 2
# upper_idx = 97

"""Minimal example:"""
num_bootstraps = 5
lower_idx = 1
upper_idx = 3

In [ ]:
unbiased_scores_boot = subsample_scores(unbiased_scores, unbiased_samples_for_biased)
lower_bound, upper_bound = bootstrap_conf_interval(trajectories,
                                                   unbiased_scores_boot,
                                                   bins,
                                                   num_bootstraps,
                                                   lower_idx,
                                                   upper_idx)

In [ ]:
plt.step(bin_centers, histogram_heights, where="mid", label="Biased")
plt.fill_between(bin_centers, 
                 lower_bound, 
                 upper_bound, 
                 step="mid", 
                 label="96% conf iterval",
                 color="C0",
                 alpha=0.5)
plt.xlabel(obs_name)
plt.ylabel("Density")
plt.yscale("log")
plt.legend()

### Wilson Interval

In [ ]:
def wilson_interval(data, bins, zscore, density=True, cumulative=False):
    n = len(data)
    n_s, _ = np.histogram(data, bins)

    n_f = n - n_s
    
    p = (n_s + 0.5 * (zscore**2)) / (n + zscore**2)
    diff = (zscore / (n + (zscore**2))) * np.sqrt(((n_s * n_f) / n) + (zscore**2 / 4))
    
    if density is True:
        upper = (p - diff) / (bins[1:] - bins[:-1])
        lower = (p + diff) / (bins[1:] - bins[:-1])
        return upper, lower
    else:
        return p - diff, p + diff
    
zscore = 2.0537 # Z score for 0.96 confidence level
lower_unbiased, upper_unbiased = wilson_interval(unbiased_samples_for_histogram,
                                                 bins, 
                                                 zscore)

In [ ]:
plt.step(bin_centers, unbiased_histogram_heights, where="mid", label="Unbiased")
plt.fill_between(bin_centers, 
                 lower_unbiased, 
                 upper_unbiased, 
                 step="mid", 
                 label="96% Wilson Interval",
                 alpha=0.5)
plt.xlabel(obs_name)
plt.yscale("log")
plt.ylabel("Density")
plt.ylim(10**-7, 1)
plt.legend()